In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Cargar el dataset y dimensiones
df = pd.read_csv('./dataset/compas-scores-two-years.csv')
print(f"Dimensiones del dataset: {df.shape}")

Dimensiones del dataset: (7214, 53)


In [8]:
# Filtros de la metodología oficial de ProPublica
df_clean = df[
    (df['days_b_screening_arrest'] <= 30) &  # Arresto no posterior a 30 días tras la evaluación
    (df['days_b_screening_arrest'] >= -30) & # Arresto no anterior a 30 días antes de la evaluación
    (df['is_recid'] != -1) &                 # Excluir casos sin datos de reincidencia
    (df['c_charge_degree'] != 'O') &         # Excluir infracciones ordinarias de tráfico
    (df['score_text'] != 'N/A')              # Excluir filas sin puntuación de riesgo calculada
].copy()

# Descartamos el ruido y nos quedamos con demografía, historial penal y las puntuaciones
columnas_relevantes = [
    'sex', 'age', 'age_cat', 'race',                      # Variables demográficas
    'juv_fel_count', 'juv_misd_count', 'juv_other_count', # Antecedentes juveniles
    'priors_count', 'c_charge_degree', 'c_charge_desc',   # Historial del caso actual y arrestos previos
    'decile_score', 'score_text',                         # Puntuaciones originales del sistema COMPAS
    'two_year_recid'                                      # Variable objetivo (Target)
]

df_seleccion = df_clean[columnas_relevantes].copy()

# Eliminar valores nulos residuales en el subconjunto final
df_final = df_seleccion.dropna()

print("Resumen de Limpieza de Datos")
print(f"Instancias originales en bruto: {df.shape[0]}")
print(f"Instancias finales limpias: {df_final.shape[0]}")
print(f"Filas descartadas por ruido o nulos: {df.shape[0] - df_final.shape[0]}")
print(f"Dimensiones del dataset: {df_final.shape}")
df_final.to_csv('dataset/compas_two_years_limpio.csv', index=False)


Resumen de Limpieza de Datos
Instancias originales en bruto: 7214
Instancias finales limpias: 6167
Filas descartadas por ruido o nulos: 1047
Dimensiones del dataset: (6167, 13)


In [11]:
print("\nDistribución de la variable objetivo (two_year_recid):")
print(df_final['two_year_recid'].value_counts())



Distribución de la variable objetivo (two_year_recid):
two_year_recid
0    3358
1    2809
Name: count, dtype: int64


In [ ]:
from ydata_profiling import ProfileReport

# Generar el reporte
# Nota: Usamos explorative=True para que calcule correlaciones avanzadas (útil para variables categóricas como la raza)
profile = ProfileReport(df, title="Auditoría Inicial COMPAS - ydata-profiling", explorative=True)

# Guardar como un archivo HTML interactivo que podéis compartir entre los 4
profile.to_file("reporte_completo_compas.html")
x
# Si estás en un notebook, puedes verlo directamente aquí (aunque suele ser pesado)
# profile.to_notebook_iframe()

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 12.86it/s]


In [5]:
# 1. Crear el reporte, pero pasándole un diccionario de configuración
profile = ProfileReport(
    df, 
    title="Auditoría COMPAS: Enfoque en Sesgo Racial",
    explorative=True, # Activa cálculos estadísticos más avanzados
    
    # 2. LA CLAVE: Forzar interacciones específicas
    interactions={
        'targets': ['decile_score', 'two_year_recid', 'race'] # Queremos que cruce TODO contra estas dos columnas
    }
)

# Guardar el nuevo reporte
profile.to_file("reporte_compas_interacciones.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 44.33it/s]
